# WiFi-CSI Human Pose Estimation

**Option 2: A practical machine-learning system**

This notebook will become the public, self-contained Assignment 2 implementation. It is intentionally not executed during the scaffold milestone.

## 1. Environment and reproducibility
Install pinned dependencies, record runtime versions, set deterministic seeds, and download verified raw datasets.

In [ ]:
# Deferred: install requirements and expose the project package.
# !pip install -r requirements.txt

## 2. Technical task definition

Training input: standardized CSI amplitude with shape `(9, 5, 30)`. Training output: 14 canonical 2D joints with confidence. Deployment output: a `(14, 2)` skeleton. The practical objective is accurate and anatomically plausible pose recovery without a camera at deployment.

## 3. Dataset provenance and inspection
Document Wi-Pose and WiMANS, validate schemas after archive review, quantify exclusions, and audit participant/environment splits. Avoid decorative EDA unrelated to the model interface.

## 4. Preprocessing and alignment
Explain complex-to-amplitude conversion, antenna-link reshaping, training-only normalization, video/CSI timing alignment, the common 14-joint map, confidence filtering, and canonical coordinates.

## 5. Model family and theory-to-code mapping
Display the residual CNN, masked autoencoder, regression head, tensor transformations, parameter counts, and the implementation corresponding to each equation.

## 6. Learning objective
Compare confidence-weighted Smooth L1 coordinate loss, bone-vector loss, and symmetry regularization. Explain why minimizing coordinate error alone may produce implausible skeletons.

## 7. Condition A — source-only strict zero-shot
Train only on Wi-Pose, select by the per-epoch monitor participant, evaluate once on the Wi-Pose final participant, freeze weights, and infer on WiMANS.

## 8. Condition B — label-free target-domain adaptation
Pretrain by masked CSI reconstruction using allowed unlabeled WiMANS environments, fine-tune using Wi-Pose labels, and evaluate on the held-out WiMANS environment. This is not strict zero-shot learning.

## 9. Training behavior
Plot training and monitor loss/metrics per epoch. Explain checkpoint selection, early stopping, generalization gaps, automatic batch sizing, mixed precision, and two-GPU DDP.

## 10. Final results and error analysis
Report NME, PCK, PCK-AUC, bone-length error, per-joint and per-activity behavior, retained pseudo-label coverage, seed mean/std, and representative success/failure skeletons.

## 11. Loss versus practical objective
Identify cases where a lower training loss does not yield a more useful pose. Compare numerical accuracy with anatomical validity and cross-domain reliability.

## 12. Limitations, future work, and implementation log
Discuss AlphaPose/RTMPose pseudo-ground-truth noise, CSI hardware and environment shift, canonical-coordinate limitations, single-person scope, failed attempts, AI assistance, verification, and remaining knowledge gaps.

## Dataset exploration checkpoint (1 September 2026)
The archives were extracted without running preprocessing or training. WiMANS contains 11,286 labelled records. Its annotation table has 17 columns: record id, label, environment, WiFi band, user count, six location fields, and six activity fields. The three environments are balanced (3,762 each), as are the two bands (5,643 each). User counts are 0: 594, 1: 3,564, and 2–5: 1,782 each. The activity fields use `nothing`, `walk`, `rotation`, `jump`, `wave`, `lie_down`, `pick_up`, `sit_down`, and `stand_up`; blank fields mark unused user slots.
WiMANS therefore provides CSI plus synchronized video and categorical scene/activity ground truth, but no native joint coordinates. Skeleton labels must be derived from video and must be treated as pseudo-ground truth. Wi-Pose remains the supervised 2D joint source for this assignment.
The following cell is the reproducible, read-only inspection used for this checkpoint.

In [ ]:
from pathlib import Path
import csv, collections
p = Path('../data/extracted/wimans/annotation.csv')
with p.open(newline='') as f:
    reader = csv.DictReader(f)
    rows = list(reader)
print(len(rows), reader.fieldnames)
for name in ('environment', 'wifi_band', 'number_of_users'):
    print(name, collections.Counter(row[name] for row in rows))
print('videos', len(list((p.parent / 'video').glob('*.mp4'))))
print('CSI files', sum(1 for x in (p.parent / 'wifi_csi').rglob('*') if x.is_file()))